# 🧱 Data Ingestion to SQL & Mongo — Annotated (Portfolio-Ready)

This notebook demonstrates a production-minded **CSV → MySQL** and **CSV → MongoDB** ingestion flow with:
- clear **separation of configuration** (via environment variables),
- **idempotent** writes (upsert / unique keys),
- **chunked** processing for large files,
- **basic validation** and **logging**,
- and guidance for moving this into a maintainable package.

## Table of Contents
1. [Project Setup & Configuration](#project-setup--configuration)
2. [Data Model & Idempotency](#data-model--idempotency)
3. [MySQL Ingestion (Chunked Upsert)](#mysql-ingestion-chunked-upsert)
4. [MongoDB Ingestion (Unique Index + Upsert)](#mongodb-ingestion-unique-index--upsert)
5. [Validation & Logging](#validation--logging)
6. [Appendix: Original Code (Unmodified)](#appendix-original-code-unmodified)

In [0]:
spark

In [0]:
dbutils.fs.ls("abfss://olistdata@olistecommdatastorage.dfs.core.windows.net/")

In [0]:
# read from bronze
df = spark.read.csv("abfss://olistdata@olistecommdatastorage.dfs.core.windows.net/bronze/olist_customers_dataset.csv", header=True, inferSchema=True)
df.show()


In [0]:
geolocation_df = spark.read.csv("abfss://olistdata@olistecommdatastorage.dfs.core.windows.net/bronze/olist_geolocation_dataset.csv", header=True, inferSchema=True)
geolocation_df.display()

### Reading the Data

In [0]:
base_path = "abfss://olistdata@olistecommdatastorage.dfs.core.windows.net/bronze/"
orders_path = base_path + "olist_orders_dataset.csv"
payments_path = base_path + "olist_order_payments_dataset.csv"
reviews_path = base_path + "olist_order_reviews_dataset.csv"
items_path = base_path + "olist_order_items_dataset.csv"
customer_path = base_path + "olist_customers_dataset.csv"
geolocation_path = base_path + "olist_geolocation_dataset.csv"
sellers_path = base_path + "olist_sellers_dataset.csv"
products_path = base_path + "olist_products_dataset.csv"

orders_df = spark.read.csv(orders_path, header=True, inferSchema=True)
payments_df = spark.read.csv(payments_path, header=True, inferSchema=True)
reviews_df = spark.read.csv(reviews_path, header=True, inferSchema=True)
items_df = spark.read.csv(items_path, header=True, inferSchema=True)
customer_df = spark.read.csv(customer_path, header=True, inferSchema=True)
geolocation_df = spark.read.csv(geolocation_path, header=True, inferSchema=True)
sellers_df = spark.read.csv(sellers_path, header=True, inferSchema=True)
products_df = spark.read.csv(products_path, header=True, inferSchema=True)

### Reading data from Pymongo

In [0]:
from pymongo import MongoClient

In [0]:
# importing module
from pymongo import MongoClient

hostname = "pl331l.h.filess.io"
database = "olistDataNoSQL_columnwhat"
port = "27018"
username = "olistDataNoSQL_columnwhat"
password = "ba2bb27af44bc36737fa8ee37ecb5e777fca8131"

uri = "mongodb://" + username + ":" + password + "@" + hostname + ":" + port + "/" + database

# Connect with the portnumber and host
client = MongoClient(uri)

# Access database
mydatabase = client[database]


In [0]:
import pandas as pd
collection = mydatabase["product_categories"]

mongo_data = pd.DataFrame(list(collection.find()))
mongo_data.head()

### Cleaning de data

In [0]:
from pyspark.sql.functions import col, to_date, datediff, current_date, when

In [0]:
def clean_dataframe(df, name):
    print(f"Cleaning dataframe: {name}")
    return df.dropDuplicates().na.drop('all')

orders_df = clean_dataframe(orders_df, "Orders")
display(orders_df)

In [0]:
# convert data columns
orders_df = orders_df.withColumn("order_purchase_timestamp", to_date(col('order_purchase_timestamp')))\
    .withColumn("order_delivered_customer_date", to_date(col('order_delivered_customer_date')))\
        .withColumn("order_estimated_delivery_date", to_date(col('order_estimated_delivery_date')))

In [0]:
# calculate delivery and time delays

orders_df = orders_df.withColumn("actual_delivery_time", datediff("order_delivered_customer_date", "order_purchase_timestamp"))\
    .withColumn("estimated_delivery_time", datediff("order_estimated_delivery_date", "order_purchase_timestamp"))\
        .withColumn("delay", col("actual_delivery_time") > col("estimated_delivery_time"))\
            .withColumn("delay", when(col("delay") == True, 1).otherwise(0))

display(orders_df)

### joining

In [0]:
orders_customers_df = orders_df.join(customer_df, on=["customer_id"], how="left")
orders_payments_df  = orders_customers_df.join(payments_df, on=["order_id"], how="left")
orders_items_df     = orders_payments_df.join(items_df, on=["order_id"], how="left")
orders_products_df  = orders_items_df.join(products_df, on=["product_id"], how="left")
final_df            = orders_products_df.join(sellers_df, on=["seller_id"], how="left")

display(final_df)


In [0]:
mongo_data.drop("_id", axis=1, inplace=True)
mongo_spark_df = spark.createDataFrame(mongo_data)
display(mongo_spark_df)


In [0]:
final_df = final_df.join(mongo_spark_df, on=["product_category_name"], how="left")
display(final_df)

In [0]:
final_df.write.mode("overwrite").parquet("abfss://olistdata@olistecommdatastorage.dfs.core.windows.net/silver/")